In [3]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import pandas as pd

ROOT = Path(r"/")

PATHS = {
    "candidates": Path(r"C:/Users/Carl/Desktop/11K CANDIDATES/Village_candidates_hh_id_obs.xlsx"),
    "hh_panel": ROOT / "Built panels" / "hh_panel_roster_9_10_11_12_13_17_18.xlsx",
    # Alternative if needed:
    # "hh_panel": ROOT / "Built panels" / "hh_panel_roster_9_10_11_12_13_17_18.xlsx",
    "agsec10_wide": ROOT / "Finished sections" / "Agriculture" / "AGSEC10_wide.csv",
}

KEYS = {
    "candidates_to_hh_panel": {
        "left": ["hh_id_obs_clean"],
        "right": ["hh_id_obs_clean"],
    },
    "hh_panel_to_agsec10": {
        "left": ["Wave_clean", "HHID_clean"],
        "right": ["Wave_clean", "HHID_clean"],
    },
}


def load_table(path, sheet_name=0, dtype=str):
    path = Path(path)

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, sheet_name=sheet_name, dtype=dtype)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, dtype=dtype)

    raise ValueError(f"Unsupported file type: {path.suffix}")


def clean_id(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s == "" or s.lower() == "nan":
        return pd.NA

    try:
        if "e" in s.lower():
            s = format(Decimal(s), "f")

        if s.endswith(".0"):
            s = s[:-2]

    except InvalidOperation:
        pass

    return s


def clean_wave(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().lower()
    s = s.replace("wave", "").replace("_", "").replace("-", "").strip()

    return int(float(s))


def add_standard_keys(df, wave_col=None, hhid_col=None, hh_id_obs_col=None):
    df = df.copy()

    if wave_col is not None and wave_col in df.columns:
        df["Wave_clean"] = df[wave_col].apply(clean_wave).astype("Int64")

    if hhid_col is not None and hhid_col in df.columns:
        df["HHID_clean"] = df[hhid_col].map(clean_id)

    if hh_id_obs_col is not None and hh_id_obs_col in df.columns:
        df["hh_id_obs_clean"] = df[hh_id_obs_col].map(clean_id)

    return df


def require_columns(df, cols, name):
    missing = [c for c in cols if c not in df.columns]

    if missing:
        print(f"WARNING: {name} is missing columns:")
        print(missing)

    return [c for c in cols if c in df.columns]


def key_report(left, right, left_keys, right_keys, right_name):
    left_key_df = left[left_keys].drop_duplicates()
    right_key_df = right[right_keys].drop_duplicates()

    check = left_key_df.merge(
        right_key_df,
        left_on=left_keys,
        right_on=right_keys,
        how="left",
        indicator=True,
    )

    print(f"\nMerge diagnostic: {right_name}")
    print("Left unique keys:", len(left_key_df))
    print("Right unique keys:", len(right_key_df))
    print(check["_merge"].value_counts(dropna=False))

    return check

In [13]:


candidates = load_table(PATHS["candidates"], sheet_name=0)

candidates = add_standard_keys(
    candidates,
    hh_id_obs_col="hh_id_obs",
)

print("Candidates shape:", candidates.shape)
display(candidates.head())



hh_panel = load_table(PATHS["hh_panel"], sheet_name=0)

hh_panel = add_standard_keys(
    hh_panel,
    wave_col="Wave",
    hhid_col="HHID",
    hh_id_obs_col="hh_id_obs",
)

hh_vars_wanted = [
    "Wave",
    "HHID",
    "hh_id_obs",
    "GSEC15A__TOTAL_HH_MEMBERS_15A",
    "GSEC12__H12Q01",
    "H11Q01",
    "GSEC10__H10Q1",
    "GSEC10__H10Q09",
    "H18Q1A", "H18Q1B", "H18Q1C", "H18Q1D",
    "H18Q4A", "H18Q4B", "H18Q4C", "H18Q4D",
    "GSEC17__H17Q9",
    "GSEC17__H17Q10",
    "GSEC17__H17Q11",
    "Wave_clean",
    "HHID_clean",
    "hh_id_obs_clean",
]

hh_vars_keep = require_columns(hh_panel, hh_vars_wanted, "HH panel")

hh_panel_small = hh_panel[hh_vars_keep].copy()

dupes = hh_panel_small[
    hh_panel_small.duplicated(["Wave_clean", "HHID_clean"], keep=False)
]

print("HH panel selected shape:", hh_panel_small.shape)
print("Duplicate Wave-HHID rows in HH panel:", len(dupes))
display(dupes.head(20))


# Expand candidates to all waves available in HH panel using hh_id_obs


key_report(
    left=candidates,
    right=hh_panel_small,
    left_keys=["hh_id_obs_clean"],
    right_keys=["hh_id_obs_clean"],
    right_name="HH panel by hh_id_obs",
)

candidate_panel = candidates.merge(
    hh_panel_small,
    on="hh_id_obs_clean",
    how="left",
    validate="m:m",
    suffixes=("", "_hhpanel"),
    indicator="hh_panel_merge_status",
)

print("Candidate panel after HH merge:", candidate_panel.shape)
display(candidate_panel["hh_panel_merge_status"].value_counts(dropna=False))
display(candidate_panel.head())


# Load AGSEC10 wide and merge using Wave + HHID

agsec10 = load_table(PATHS["agsec10_wide"])

agsec10 = add_standard_keys(
    agsec10,
    wave_col="Wave",
    hhid_col="HHID",
)

agsec10_vars_wanted = [
    "Wave",
    "HHID",
    "AGSEC10_ANY",
    "AGSEC10_PROD",
    "AGSEC10_PRICES",
    "AGSEC10_PROC",
    "Wave_clean",
    "HHID_clean",
]

agsec10_vars_keep = require_columns(agsec10, agsec10_vars_wanted, "AGSEC10 wide")

agsec10_small = agsec10[agsec10_vars_keep].copy()

agsec10_small = (
    agsec10_small
    .drop_duplicates(["Wave_clean", "HHID_clean"])
)

key_report(
    left=candidate_panel,
    right=agsec10_small,
    left_keys=["Wave_clean", "HHID_clean"],
    right_keys=["Wave_clean", "HHID_clean"],
    right_name="AGSEC10 wide by Wave-HHID",
)

candidate_panel = candidate_panel.merge(
    agsec10_small.drop(columns=["Wave", "HHID"], errors="ignore"),
    on=["Wave_clean", "HHID_clean"],
    how="left",
    validate="m:1",
    indicator="agsec10_merge_status",
)

print("Candidate panel after AGSEC10 merge:", candidate_panel.shape)
display(candidate_panel["agsec10_merge_status"].value_counts(dropna=False))
display(candidate_panel.head())


# AGSEC10_ANY should be 0 for households with no AGSEC10 match.
# The other AGSEC10 columns stay missing unless a household received extension service.
if "AGSEC10_ANY" in candidate_panel.columns:
    candidate_panel["AGSEC10_ANY"] = pd.to_numeric(
        candidate_panel["AGSEC10_ANY"],
        errors="coerce"
    ).fillna(0)

candidate_panel = candidate_panel.drop(columns=["agsec10_merge_status"])

Candidates shape: (458039, 4)


,index,name_11k,hh_id_obs,hh_id_obs_clean
0,3,Ziru,7000197,7000197
1,3,Ziru,7000203,7000203
2,3,Ziru,7000245,7000245
3,3,Ziru,7000264,7000264
4,3,Ziru,7000317,7000317


HH panel selected shape: (21239, 22)
Duplicate Wave-HHID rows in HH panel: 0


,Wave,HHID,hh_id_obs,GSEC15A__TOTAL_HH_MEMBERS_15A,GSEC12__H12Q01,H11Q01,GSEC10__H10Q1,GSEC10__H10Q09,H18Q1A,H18Q1B,...,H18Q4A,H18Q4B,H18Q4C,H18Q4D,GSEC17__H17Q9,GSEC17__H17Q10,GSEC17__H17Q11,Wave_clean,HHID_clean,hh_id_obs_clean



Merge diagnostic: HH panel by hh_id_obs
Left unique keys: 3102
Right unique keys: 5654
_merge
both          3101
left_only        1
right_only       0
Name: count, dtype: int64
Candidate panel after HH merge: (1866868, 26)


hh_panel_merge_status
both          1866850
left_only          18
right_only          0
Name: count, dtype: int64

,index,name_11k,hh_id_obs,hh_id_obs_clean,Wave,HHID,hh_id_obs_hhpanel,GSEC15A__TOTAL_HH_MEMBERS_15A,GSEC12__H12Q01,H11Q01,...,H18Q4A,H18Q4B,H18Q4C,H18Q4D,GSEC17__H17Q9,GSEC17__H17Q10,GSEC17__H17Q11,Wave_clean,HHID_clean,hh_panel_merge_status
0,3,Ziru,7000197,7000197,1,1021002204,7000197,3.0,1,4.0,...,NaN,NaN,10,NaN,1,"A,B,C,H",E,1,1021002204,both
1,3,Ziru,7000197,7000197,2,1021002204,7000197,3.0,1,3.0,...,NaN,NaN,30,NaN,0,NaN,NaN,2,1021002204,both
2,3,Ziru,7000197,7000197,3,1021002204,7000197,3.0,1,3.0,...,NaN,NaN,2,0,0,NaN,NaN,3,1021002204,both
3,3,Ziru,7000197,7000197,4,H02304-04-01,7000197,2.0,1,4.0,...,NaN,NaN,NaN,NaN,0,NaN,NaN,4,H02304-04-01,both
4,3,Ziru,7000197,7000197,5,H0230401,7000197,2.0,1,4.0,...,NaN,NaN,NaN,NaN,0,NaN,NaN,5,H0230401,both



Merge diagnostic: AGSEC10 wide by Wave-HHID
Left unique keys: 12429
Right unique keys: 2973
_merge
left_only     11025
both           1404
right_only        0
Name: count, dtype: int64
Candidate panel after AGSEC10 merge: (1866868, 31)


agsec10_merge_status
left_only     1686339
both           180529
right_only          0
Name: count, dtype: int64

,index,name_11k,hh_id_obs,hh_id_obs_clean,Wave,HHID,hh_id_obs_hhpanel,GSEC15A__TOTAL_HH_MEMBERS_15A,GSEC12__H12Q01,H11Q01,...,GSEC17__H17Q10,GSEC17__H17Q11,Wave_clean,HHID_clean,hh_panel_merge_status,AGSEC10_ANY,AGSEC10_PROD,AGSEC10_PRICES,AGSEC10_PROC,agsec10_merge_status
0,3,Ziru,7000197,7000197,1,1021002204,7000197,3.0,1,4.0,...,"A,B,C,H",E,1,1021002204,both,NaN,NaN,NaN,NaN,left_only
1,3,Ziru,7000197,7000197,2,1021002204,7000197,3.0,1,3.0,...,NaN,NaN,2,1021002204,both,NaN,NaN,NaN,NaN,left_only
2,3,Ziru,7000197,7000197,3,1021002204,7000197,3.0,1,3.0,...,NaN,NaN,3,1021002204,both,NaN,NaN,NaN,NaN,left_only
3,3,Ziru,7000197,7000197,4,H02304-04-01,7000197,2.0,1,4.0,...,NaN,NaN,4,H02304-04-01,both,NaN,NaN,NaN,NaN,left_only
4,3,Ziru,7000197,7000197,5,H0230401,7000197,2.0,1,4.0,...,NaN,NaN,5,H0230401,both,NaN,NaN,NaN,NaN,left_only


In [14]:


AG3_FILE = (
    ROOT
    / "Built panels"
    / "AG3_inputs_aggregated.xlsx"
)

ag3 = load_table(AG3_FILE, sheet_name=0)

ag3 = add_standard_keys(
    ag3,
    wave_col="Wave",
    hhid_col="HHID",
)

ag3["VISIT"] = pd.to_numeric(ag3["VISIT"], errors="coerce").astype("Int64")

ag3_value_cols = [
    "A3Q4_average_use",
    "A3Q14_average_use",
    "A3Q26_average_use",
    "A3AQ38_average",
    "A3Q39_average",
    "A3Q41_any_labor",
    "A3Q43_sum",
]

ag3_value_cols = require_columns(ag3, ag3_value_cols, "AG3 inputs")

for col in ag3_value_cols:
    ag3[col] = pd.to_numeric(ag3[col], errors="coerce")


ag3_dupes = ag3[
    ag3.duplicated(["Wave_clean", "HHID_clean", "VISIT"], keep=False)
].copy()

print("AG3 duplicate Wave-HHID-VISIT rows:", len(ag3_dupes))
display(ag3_dupes.head(20))


ag3_collapsed = (
    ag3
    .groupby(["Wave_clean", "HHID_clean", "VISIT"], as_index=False)
    .agg({
        col: "mean" if col != "A3Q43_sum" else "sum"
        for col in ag3_value_cols
    })
)


ag3_name_map = {
    "A3Q4_average_use": "A3Q4",
    "A3Q14_average_use": "A3Q14",
    "A3Q26_average_use": "A3Q26",
    "A3AQ38_average": "A3AQ38",
    "A3Q39_average": "A3Q39",
    "A3Q41_any_labor": "A3Q41",
    "A3Q43_sum": "A3Q43",
}

ag3_wide_parts = []

for source_col, output_prefix in ag3_name_map.items():
    if source_col not in ag3_collapsed.columns:
        continue

    part = (
        ag3_collapsed
        .pivot(
            index=["Wave_clean", "HHID_clean"],
            columns="VISIT",
            values=source_col
        )
        .reindex(columns=[1, 2])
    )

    part.columns = [
        f"{output_prefix}_{int(visit)}"
        for visit in part.columns
    ]

    ag3_wide_parts.append(part)

ag3_wide = pd.concat(ag3_wide_parts, axis=1).reset_index()

print("AG3 wide shape:", ag3_wide.shape)
display(ag3_wide.head())

before_rows = len(candidate_panel)

candidate_panel = candidate_panel.merge(
    ag3_wide,
    on=["Wave_clean", "HHID_clean"],
    how="left",
    validate="m:1",
    indicator="ag3_merge_status",
)

after_rows = len(candidate_panel)

if before_rows != after_rows:
    raise ValueError(f"Row count changed during AG3 merge: {before_rows} -> {after_rows}")

print("AG3 merge status:")
display(candidate_panel["ag3_merge_status"].value_counts(dropna=False))

candidate_panel = candidate_panel.drop(columns=["ag3_merge_status"])

display(candidate_panel.head())

AG3 duplicate Wave-HHID-VISIT rows: 0


,Wave,HHID,VISIT,A3Q4_average_use,A3Q14_average_use,A3Q26_average_use,A3AQ38_average,A3Q39_average,A3Q41_any_labor,A3Q43_sum,Wave_clean,HHID_clean


AG3 wide shape: (16900, 16)


,Wave_clean,HHID_clean,A3Q4_1,A3Q4_2,A3Q14_1,A3Q14_2,A3Q26_1,A3Q26_2,A3AQ38_1,A3AQ38_2,A3Q39_1,A3Q39_2,A3Q41_1,A3Q41_2,A3Q43_1,A3Q43_2
0,1,1013000204,NaN,0.0,NaN,0.0,NaN,0.0,NaN,1.000000,NaN,40.000000,0.0,1.0,0.0,4000.0
1,1,1021000108,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.666667,28.0,15.333333,1.0,1.0,30000.0,80000.0
2,1,1021000113,1.0,0.8,0.0,0.0,0.0,0.0,2.0,2.000000,36.0,8.400000,1.0,1.0,50000.0,55600.0
3,1,1021000408,0.0,NaN,0.0,NaN,0.0,NaN,1.0,NaN,16.0,NaN,0.0,0.0,0.0,0.0
4,1,1021000710,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.000000,NaN,21.000000,1.0,1.0,75000.0,20000.0


AG3 merge status:


ag3_merge_status
both          1378351
left_only      488517
right_only          0
Name: count, dtype: int64

,index,name_11k,hh_id_obs,hh_id_obs_clean,Wave,HHID,hh_id_obs_hhpanel,GSEC15A__TOTAL_HH_MEMBERS_15A,GSEC12__H12Q01,H11Q01,...,A3Q26_1,A3Q26_2,A3AQ38_1,A3AQ38_2,A3Q39_1,A3Q39_2,A3Q41_1,A3Q41_2,A3Q43_1,A3Q43_2
0,3,Ziru,7000197,7000197,1,1021002204,7000197,3.0,1,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,Ziru,7000197,7000197,2,1021002204,7000197,3.0,1,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Ziru,7000197,7000197,3,1021002204,7000197,3.0,1,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Ziru,7000197,7000197,4,H02304-04-01,7000197,2.0,1,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,Ziru,7000197,7000197,5,H0230401,7000197,2.0,1,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:

# Add geonameid, rainfall, and density covariates to candidate_panel


from pathlib import Path
import pandas as pd

CAND_ROOT = Path(r"C:/Users/Carl/Desktop/11K CANDIDATES")
CAND_30KM = CAND_ROOT / "data" / "Candidate(30KM)"

FILES = {
    "merge_indices": CAND_ROOT / "merge indices.xlsx",
    "rainfall": CAND_30KM / "11K_weighted_30km_mean_seasonal_rainfall_2000_2010.xlsx",
    "density": CAND_30KM / "CANDIDATES_density_means_30km_.xlsx",
}


def resolve_file(path):
    path = Path(path)

    if path.exists():
        return path

    if path.suffix.lower() == ".xlsx":
        csv_path = path.with_suffix(".csv")
        if csv_path.exists():
            return csv_path

    raise FileNotFoundError(path)


def load_covariate_file(path):
    path = resolve_file(path)

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, dtype=str)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, dtype=str)

    raise ValueError(f"Unsupported file type: {path.suffix}")


def clean_merge_key(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s == "" or s.lower() == "nan":
        return pd.NA

    if s.endswith(".0"):
        s = s[:-2]

    return s


def merge_m1_with_report(left, right, on, name):
    before_rows = len(left)

    out = left.merge(
        right,
        on=on,
        how="left",
        validate="m:1",
        indicator=f"{name}_merge_status",
    )

    after_rows = len(out)

    if before_rows != after_rows:
        raise ValueError(f"{name} merge changed rows: {before_rows} -> {after_rows}")

    print(f"\n{name} merge status:")
    display(out[f"{name}_merge_status"].value_counts(dropna=False))

    out = out.drop(columns=[f"{name}_merge_status"])

    return out




candidate_panel = candidate_panel.copy()


cols_to_replace = [
    "geonameid",
    "Mean_season_1",
    "Mean_season_2",
    "density_2010",
    "Trend",
]

candidate_panel = candidate_panel.drop(
    columns=[c for c in cols_to_replace if c in candidate_panel.columns],
    errors="ignore"
)

if "index" in candidate_panel.columns:
    candidate_panel["index_clean"] = candidate_panel["index"].map(clean_merge_key)
elif "Index" in candidate_panel.columns:
    candidate_panel["index_clean"] = candidate_panel["Index"].map(clean_merge_key)
else:
    raise KeyError("candidate_panel needs an index/Index column for these merges.")


# 1. merge indices.xlsx: index -> geonameid

merge_indices = load_covariate_file(FILES["merge_indices"])

merge_indices.columns = merge_indices.columns.str.strip()

merge_indices = merge_indices.rename(
    columns={
        "Index": "index",
        "index": "index",
        "geonameid": "geonameid",
    }
)

merge_indices["index_clean"] = merge_indices["index"].map(clean_merge_key)
merge_indices["geonameid_clean"] = merge_indices["geonameid"].map(clean_merge_key)

merge_indices_map = (
    merge_indices[["index_clean", "geonameid", "geonameid_clean"]]
    .drop_duplicates()
)

geo_conflicts = (
    merge_indices_map
    .groupby("index_clean")["geonameid_clean"]
    .nunique(dropna=True)
    .reset_index(name="n_geonameid")
    .query("n_geonameid > 1")
)

print("Index -> geonameid conflicts:", len(geo_conflicts))
display(geo_conflicts.head(20))

merge_indices_map = (
    merge_indices_map
    .drop_duplicates("index_clean")
)

candidate_panel = merge_m1_with_report(
    candidate_panel,
    merge_indices_map,
    on="index_clean",
    name="geonameid"
)


# 2. Rainfall: geonameid -> Mean_season_1, Mean_season_2

rainfall = load_covariate_file(FILES["rainfall"])

rainfall.columns = rainfall.columns.str.strip()

rainfall["geonameid_clean"] = rainfall["geonameid"].map(clean_merge_key)

rainfall_map = (
    rainfall[["geonameid_clean", "Mean_season_1", "Mean_season_2"]]
    .drop_duplicates()
)

rain_conflicts = (
    rainfall_map
    .groupby("geonameid_clean")[["Mean_season_1", "Mean_season_2"]]
    .nunique(dropna=True)
    .reset_index()
    .query("Mean_season_1 > 1 or Mean_season_2 > 1")
)

print("Rainfall geonameid conflicts:", len(rain_conflicts))
display(rain_conflicts.head(20))

rainfall_map = rainfall_map.drop_duplicates("geonameid_clean")

candidate_panel = merge_m1_with_report(
    candidate_panel,
    rainfall_map,
    on="geonameid_clean",
    name="rainfall"
)



# 3. Density: index -> density_2010, Trend

density = load_covariate_file(FILES["density"])

density.columns = density.columns.str.strip()

density["index_clean"] = density["index"].map(clean_merge_key)

density_map = (
    density[["index_clean", "density_2010", "Trend"]]
    .drop_duplicates()
)

density_conflicts = (
    density_map
    .groupby("index_clean")[["density_2010", "Trend"]]
    .nunique(dropna=True)
    .reset_index()
    .query("density_2010 > 1 or Trend > 1")
)

print("Density index conflicts:", len(density_conflicts))
display(density_conflicts.head(20))

density_map = density_map.drop_duplicates("index_clean")

candidate_panel = merge_m1_with_report(
    candidate_panel,
    density_map,
    on="index_clean",
    name="density"
)



for col in ["Mean_season_1", "Mean_season_2", "density_2010", "Trend"]:
    if col in candidate_panel.columns:
        candidate_panel[col] = pd.to_numeric(candidate_panel[col], errors="coerce")

print("\nCandidate panel shape after covariates:", candidate_panel.shape)
display(candidate_panel.head())

Index -> geonameid conflicts: 0


,index_clean,n_geonameid



geonameid merge status:


geonameid_merge_status
both          1866868
left_only           0
right_only          0
Name: count, dtype: int64

Rainfall geonameid conflicts: 0


...

,index_clean,density_2010,Trend



density merge status:


density_merge_status
both          1866868
left_only           0
right_only          0
Name: count, dtype: int64


Candidate panel shape after covariates: (1866868, 51)


,index,name_11k,hh_id_obs,hh_id_obs_clean,Wave,HHID,hh_id_obs_hhpanel,GSEC15A__TOTAL_HH_MEMBERS_15A,GSEC12__H12Q01,H11Q01,...,A3Q41_2,A3Q43_1,A3Q43_2,index_clean,geonameid,geonameid_clean,Mean_season_1,Mean_season_2,density_2010,Trend
0,3,Ziru,7000197,7000197,1,1021002204,7000197,3.0,1,4.0,...,NaN,NaN,NaN,3,225803,225803,777.952856,630.530417,1463.404023,0.039802
1,3,Ziru,7000197,7000197,2,1021002204,7000197,3.0,1,3.0,...,NaN,NaN,NaN,3,225803,225803,777.952856,630.530417,1463.404023,0.039802
2,3,Ziru,7000197,7000197,3,1021002204,7000197,3.0,1,3.0,...,NaN,NaN,NaN,3,225803,225803,777.952856,630.530417,1463.404023,0.039802
3,3,Ziru,7000197,7000197,4,H02304-04-01,7000197,2.0,1,4.0,...,NaN,NaN,NaN,3,225803,225803,777.952856,630.530417,1463.404023,0.039802
4,3,Ziru,7000197,7000197,5,H0230401,7000197,2.0,1,4.0,...,NaN,NaN,NaN,3,225803,225803,777.952856,630.530417,1463.404023,0.039802


In [16]:

# create derived variables and aggregate candidate_panel to village-wave level


candidate_panel_agg_source = candidate_panel.copy()


if "index" in candidate_panel_agg_source.columns:
    village_key = "index"
elif "Index" in candidate_panel_agg_source.columns:
    village_key = "Index"
else:
    raise KeyError("No village index column found. Expected 'index' or 'Index'.")

candidate_panel_agg_source[village_key] = (
    candidate_panel_agg_source[village_key]
    .astype("string")
    .str.strip()
)

if "Wave" not in candidate_panel_agg_source.columns:
    raise KeyError("candidate_panel needs a Wave column.")

candidate_panel_agg_source["Wave"] = pd.to_numeric(
    candidate_panel_agg_source["Wave"],
    errors="coerce"
).astype("Int64")

numeric_cols = [
    "A3AQ38_1",
    "A3AQ38_2",
    "GSEC15A__TOTAL_HH_MEMBERS_15A",
    "H11Q01",
]

for col in numeric_cols:
    if col in candidate_panel_agg_source.columns:
        candidate_panel_agg_source[col] = pd.to_numeric(
            candidate_panel_agg_source[col],
            errors="coerce"
        )


if "A3AQ38_1" in candidate_panel_agg_source.columns:
    outlier_mask = candidate_panel_agg_source["A3AQ38_1"].eq(8336.3)

    print("A3AQ38_1 outlier cells set to missing:", int(outlier_mask.sum()))

    candidate_panel_agg_source.loc[outlier_mask, "A3AQ38_1"] = pd.NA


hh_size_col = "GSEC15A__TOTAL_HH_MEMBERS_15A"

if hh_size_col in candidate_panel_agg_source.columns:
    hh_size = candidate_panel_agg_source[hh_size_col]

    for visit in [1, 2]:
        src = f"A3AQ38_{visit}"
        out = f"share_hh_working_{visit}"

        if src in candidate_panel_agg_source.columns:
            candidate_panel_agg_source[out] = (
                candidate_panel_agg_source[src] / hh_size
            )

            candidate_panel_agg_source.loc[
                hh_size.le(0) | hh_size.isna(),
                out
            ] = pd.NA



if "H11Q01" in candidate_panel_agg_source.columns:
    candidate_panel_agg_source["subsistence_agriculture"] = (
        candidate_panel_agg_source["H11Q01"].eq(4)
    ).astype("Int64")

-

non_numeric_keep = [
    village_key,
    "name_11k",
    "geonameid",
]

for col in candidate_panel_agg_source.columns:
    if col not in non_numeric_keep:
        converted = pd.to_numeric(candidate_panel_agg_source[col], errors="coerce")

        # Only replace if at least one value converts, so text columns stay text
        if converted.notna().any():
            candidate_panel_agg_source[col] = converted



agsec10_rate_cols = [
    "AGSEC10_ANY",
    "AGSEC10_PROD",
    "AGSEC10_PRICES",
    "AGSEC10_PROC",
]

for col in agsec10_rate_cols:
    if col in candidate_panel_agg_source.columns:
        candidate_panel_agg_source[col] = pd.to_numeric(
            candidate_panel_agg_source[col],
            errors="coerce"
        )



group_keys = [village_key, "Wave"]

numeric_for_mean = (
    candidate_panel_agg_source
    .select_dtypes(include="number")
    .columns
    .tolist()
)

exclude_from_mean = {
    "HHID",
    "HHID_clean",
    "hh_id_obs",
    "hh_id_obs_clean",
    "hh_id_obs_hhpanel",
    "Wave",
    "Wave_clean",
    "geonameid_clean",
}

numeric_for_mean = [
    c for c in numeric_for_mean
    if c not in exclude_from_mean
    and c != village_key
]

agg_spec = {
    col: "mean"
    for col in numeric_for_mean
}

if "name_11k" in candidate_panel_agg_source.columns:
    agg_spec["name_11k"] = "first"

if "geonameid" in candidate_panel_agg_source.columns:
    agg_spec["geonameid"] = "first"

if "hh_id_obs_clean" in candidate_panel_agg_source.columns:
    agg_spec["hh_id_obs_clean"] = pd.Series.nunique
elif "hh_id_obs" in candidate_panel_agg_source.columns:
    agg_spec["hh_id_obs"] = pd.Series.nunique
else:
    raise KeyError("Need hh_id_obs or hh_id_obs_clean to count households.")

candidate_panel_village_wave = (
    candidate_panel_agg_source
    .groupby(group_keys, as_index=False)
    .agg(agg_spec)
)

if "subsistence_agriculture" in candidate_panel_village_wave.columns:
    candidate_panel_village_wave = candidate_panel_village_wave.rename(
        columns={"subsistence_agriculture": "avg_subsistence"}
    )

if "hh_id_obs_clean" in candidate_panel_village_wave.columns:
    candidate_panel_village_wave = candidate_panel_village_wave.rename(
        columns={"hh_id_obs_clean": "n_HHID_wave"}
    )

if "hh_id_obs" in candidate_panel_village_wave.columns:
    candidate_panel_village_wave = candidate_panel_village_wave.rename(
        columns={"hh_id_obs": "n_HHID_wave"}
    )

front_cols = [
    village_key,
    "name_11k",
    "geonameid",
    "Wave",
    "n_HHID_wave",
    "avg_subsistence",
    "AGSEC10_ANY",
    "AGSEC10_PROD",
    "AGSEC10_PRICES",
    "AGSEC10_PROC",
    "share_hh_working_1",
    "share_hh_working_2",
]

front_cols = [c for c in front_cols if c in candidate_panel_village_wave.columns]
other_cols = [c for c in candidate_panel_village_wave.columns if c not in front_cols]

candidate_panel_village_wave = candidate_panel_village_wave[front_cols + other_cols]

print("Original candidate_panel rows:", len(candidate_panel))
print("Village-wave rows:", len(candidate_panel_village_wave))
print("Village-wave columns:", len(candidate_panel_village_wave.columns))

display(candidate_panel_village_wave.head())

A3AQ38_1 outlier cells set to missing: 0
Original candidate_panel rows: 1866868
Village-wave rows: 35879
Village-wave columns: 45


,index,name_11k,geonameid,Wave,n_HHID_wave,avg_subsistence,AGSEC10_ANY,AGSEC10_PROD,AGSEC10_PRICES,AGSEC10_PROC,...,A3Q39_2,A3Q41_1,A3Q41_2,A3Q43_1,A3Q43_2,index_clean,Mean_season_1,Mean_season_2,density_2010,Trend
0,10,Ziba,225826,1,46,0.369565,0.130435,1.0,0.666667,0.166667,...,34.253009,0.305556,0.166667,11977.777778,14288.888889,10.0,744.170521,653.835264,277.865119,0.025182
1,10,Ziba,225826,2,44,0.363636,0.090909,1.0,0.500000,0.250000,...,41.490000,0.205882,0.193548,21323.529412,10258.064516,10.0,744.170521,653.835264,277.865119,0.025182
2,10,Ziba,225826,3,43,0.395349,0.162791,1.0,0.571429,0.428571,...,37.545873,0.241379,0.200000,35551.724138,23500.000000,10.0,744.170521,653.835264,277.865119,0.025182
3,10,Ziba,225826,4,55,0.327273,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10.0,744.170521,653.835264,277.865119,0.025182
4,10,Ziba,225826,5,59,0.322034,0.118644,1.0,0.571429,0.285714,...,51.582540,0.541667,0.355556,52395.833333,53855.555556,10.0,744.170521,653.835264,277.865119,0.025182


In [17]:
OUT = Path(r"C:/Users/Carl/Desktop/11K CANDIDATES")
out_file = OUT / "Village_candidates_panel_v2.xlsx"

candidate_panel_village_wave.to_excel(out_file, index=False)

print("Saved:", out_file)

Saved: C:\Users\Carl\Desktop\11K CANDIDATES\Village_candidates_panel_v2.xlsx


In [18]:

# CSB treatment-site panel: household merge + AGSEC10 + AG3 + covariates

from pathlib import Path
import pandas as pd

ROOT = Path(r"/")

CSB_ROOT = Path(r"C:/Users/Carl/Desktop/CSB Baseline data")
CSB_30KM = CSB_ROOT / "30km treatment area"

CSB_PATHS = {
    "csb_hh": CSB_ROOT / "CSB_hh_id_obs.xlsx",
    "hh_panel": ROOT / "Built panels" / "hh_panel_roster_9_10_11_12_13_17_18.xlsx",
    "agsec10_wide": ROOT / "Finished sections" / "Agriculture" / "AGSEC10_wide.csv",
    "ag3_inputs": ROOT / "Built panels" / "AG3_inputs_aggregated.xlsx",
    "rainfall": CSB_30KM / "CSB_weighted_30km_mean_seasonal_rainfall_2000_2010.csv",
    "density": CSB_30KM / "csb_density_means_30km_fixed.xlsx",
}


def clean_csb_name(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()
    s = " ".join(s.split())

    return s


def sum_with_missing(s):
    return s.sum(min_count=1)


csb_hh = load_table(CSB_PATHS["csb_hh"], sheet_name=0)

csb_hh.columns = csb_hh.columns.str.strip()

csb_hh = add_standard_keys(
    csb_hh,
    hh_id_obs_col="hh_id_obs",
)

csb_hh["CSB_clean"] = csb_hh["Seed Bank"].map(clean_csb_name)

print("CSB HH list shape:", csb_hh.shape)
display(csb_hh.head())


hh_panel = load_table(CSB_PATHS["hh_panel"], sheet_name=0)

hh_panel = add_standard_keys(
    hh_panel,
    wave_col="Wave",
    hhid_col="HHID",
    hh_id_obs_col="hh_id_obs",
)

hh_vars_wanted = [
    "Wave",
    "HHID",
    "hh_id_obs",
    "GSEC15A__TOTAL_HH_MEMBERS_15A",
    "GSEC12__H12Q01",
    "H11Q01",
    "GSEC10__H10Q1",
    "GSEC10__H10Q09",
    "H18Q1A", "H18Q1B", "H18Q1C", "H18Q1D",
    "H18Q4A", "H18Q4B", "H18Q4C", "H18Q4D",
    "GSEC17__H17Q9",
    "GSEC17__H17Q10",
    "GSEC17__H17Q11",
    "Wave_clean",
    "HHID_clean",
    "hh_id_obs_clean",
]

hh_vars_keep = require_columns(hh_panel, hh_vars_wanted, "HH panel")
hh_panel_small = hh_panel[hh_vars_keep].copy()

key_report(
    left=csb_hh,
    right=hh_panel_small,
    left_keys=["hh_id_obs_clean"],
    right_keys=["hh_id_obs_clean"],
    right_name="HH panel by hh_id_obs",
)

csb_panel = csb_hh.merge(
    hh_panel_small,
    on="hh_id_obs_clean",
    how="left",
    validate="m:m",
    suffixes=("", "_hhpanel"),
    indicator="hh_panel_merge_status",
)

print("CSB panel after HH merge:", csb_panel.shape)
display(csb_panel["hh_panel_merge_status"].value_counts(dropna=False))


agsec10 = load_table(CSB_PATHS["agsec10_wide"])

agsec10 = add_standard_keys(
    agsec10,
    wave_col="Wave",
    hhid_col="HHID",
)

agsec10_vars_wanted = [
    "Wave",
    "HHID",
    "AGSEC10_ANY",
    "AGSEC10_PROD",
    "AGSEC10_PRICES",
    "AGSEC10_PROC",
    "Wave_clean",
    "HHID_clean",
]

agsec10_vars_keep = require_columns(agsec10, agsec10_vars_wanted, "AGSEC10 wide")
agsec10_small = agsec10[agsec10_vars_keep].drop_duplicates(["Wave_clean", "HHID_clean"])

csb_panel = csb_panel.merge(
    agsec10_small.drop(columns=["Wave", "HHID"], errors="ignore"),
    on=["Wave_clean", "HHID_clean"],
    how="left",
    validate="m:1",
    indicator="agsec10_merge_status",
)

print("AGSEC10 merge status:")
display(csb_panel["agsec10_merge_status"].value_counts(dropna=False))

if "AGSEC10_ANY" in csb_panel.columns:
    csb_panel["AGSEC10_ANY"] = pd.to_numeric(csb_panel["AGSEC10_ANY"], errors="coerce").fillna(0)

csb_panel = csb_panel.drop(columns=["agsec10_merge_status"])

ag3 = load_table(CSB_PATHS["ag3_inputs"], sheet_name=0)

ag3 = add_standard_keys(
    ag3,
    wave_col="Wave",
    hhid_col="HHID",
)

ag3["VISIT"] = pd.to_numeric(ag3["VISIT"], errors="coerce").astype("Int64")

ag3_value_cols = [
    "A3Q4_average_use",
    "A3Q14_average_use",
    "A3Q26_average_use",
    "A3AQ38_average",
    "A3Q39_average",
    "A3Q41_any_labor",
    "A3Q43_sum",
]

ag3_value_cols = require_columns(ag3, ag3_value_cols, "AG3 inputs")

for col in ag3_value_cols:
    ag3[col] = pd.to_numeric(ag3[col], errors="coerce")

ag3_collapsed = (
    ag3
    .groupby(["Wave_clean", "HHID_clean", "VISIT"], as_index=False)
    .agg({
        col: "mean" if col != "A3Q43_sum" else "sum"
        for col in ag3_value_cols
    })
)

ag3_name_map = {
    "A3Q4_average_use": "A3Q4",
    "A3Q14_average_use": "A3Q14",
    "A3Q26_average_use": "A3Q26",
    "A3AQ38_average": "A3AQ38",
    "A3Q39_average": "A3Q39",
    "A3Q41_any_labor": "A3Q41",
    "A3Q43_sum": "A3Q43",
}

ag3_wide_parts = []

for source_col, output_prefix in ag3_name_map.items():
    part = (
        ag3_collapsed
        .pivot(
            index=["Wave_clean", "HHID_clean"],
            columns="VISIT",
            values=source_col
        )
        .reindex(columns=[1, 2])
    )

    part.columns = [
        f"{output_prefix}_{int(visit)}"
        for visit in part.columns
    ]

    ag3_wide_parts.append(part)

ag3_wide = pd.concat(ag3_wide_parts, axis=1).reset_index()

csb_panel = csb_panel.merge(
    ag3_wide,
    on=["Wave_clean", "HHID_clean"],
    how="left",
    validate="m:1",
    indicator="ag3_merge_status",
)

print("AG3 merge status:")
display(csb_panel["ag3_merge_status"].value_counts(dropna=False))

csb_panel = csb_panel.drop(columns=["ag3_merge_status"])


rainfall = load_table(CSB_PATHS["rainfall"])

rainfall.columns = rainfall.columns.str.strip()
rainfall["CSB_clean"] = rainfall["CSB"].map(clean_csb_name)

rainfall_map = (
    rainfall[["CSB_clean", "Mean_season_1", "Mean_season_2"]]
    .drop_duplicates()
    .drop_duplicates("CSB_clean")
)

csb_panel = merge_m1_with_report(
    csb_panel,
    rainfall_map,
    on="CSB_clean",
    name="csb_rainfall"
)

density = load_table(CSB_PATHS["density"], sheet_name=0)

density.columns = density.columns.str.strip()
density["CSB_clean"] = density["CSB"].map(clean_csb_name)

density_map = (
    density[["CSB_clean", "density_2010", "Trend"]]
    .drop_duplicates()
    .drop_duplicates("CSB_clean")
)

csb_panel = merge_m1_with_report(
    csb_panel,
    density_map,
    on="CSB_clean",
    name="csb_density"
)

for col in ["Mean_season_1", "Mean_season_2", "density_2010", "Trend"]:
    if col in csb_panel.columns:
        csb_panel[col] = pd.to_numeric(csb_panel[col], errors="coerce")

csb_agg_source = csb_panel.copy()

needed_numeric = [
    "A3AQ38_1",
    "A3AQ38_2",
    "GSEC15A__TOTAL_HH_MEMBERS_15A",
    "H11Q01",
    "AGSEC10_ANY",
    "AGSEC10_PROD",
    "AGSEC10_PRICES",
    "AGSEC10_PROC",
]

for col in needed_numeric:
    if col in csb_agg_source.columns:
        csb_agg_source[col] = pd.to_numeric(csb_agg_source[col], errors="coerce")

if "A3AQ38_1" in csb_agg_source.columns:
    outlier_mask = csb_agg_source["A3AQ38_1"].eq(8336.3)
    print("A3AQ38_1 outlier cells set to missing:", int(outlier_mask.sum()))
    csb_agg_source.loc[outlier_mask, "A3AQ38_1"] = pd.NA

hh_size_col = "GSEC15A__TOTAL_HH_MEMBERS_15A"

if hh_size_col in csb_agg_source.columns:
    hh_size = csb_agg_source[hh_size_col]

    for visit in [1, 2]:
        src = f"A3AQ38_{visit}"
        out = f"share_hh_working_{visit}"

        if src in csb_agg_source.columns:
            csb_agg_source[out] = csb_agg_source[src] / hh_size
            csb_agg_source.loc[hh_size.le(0) | hh_size.isna(), out] = pd.NA

if "H11Q01" in csb_agg_source.columns:
    csb_agg_source["subsistence_agriculture"] = (
        csb_agg_source["H11Q01"].eq(4)
    ).astype("Int64")

non_numeric_keep = ["Seed Bank", "CSB_clean"]

for col in csb_agg_source.columns:
    if col not in non_numeric_keep:
        converted = pd.to_numeric(csb_agg_source[col], errors="coerce")

        if converted.notna().any():
            csb_agg_source[col] = converted

group_keys = ["CSB_clean", "Wave"]

numeric_for_mean = (
    csb_agg_source
    .select_dtypes(include="number")
    .columns
    .tolist()
)

exclude_from_mean = {
    "HHID",
    "HHID_clean",
    "hh_id_obs",
    "hh_id_obs_clean",
    "hh_id_obs_hhpanel",
    "Wave",
    "Wave_clean",
}

numeric_for_mean = [
    c for c in numeric_for_mean
    if c not in exclude_from_mean
]

agg_spec = {col: "mean" for col in numeric_for_mean}

agg_spec["Seed Bank"] = "first"

if "hh_id_obs_clean" in csb_agg_source.columns:
    agg_spec["hh_id_obs_clean"] = pd.Series.nunique
elif "hh_id_obs" in csb_agg_source.columns:
    agg_spec["hh_id_obs"] = pd.Series.nunique
else:
    raise KeyError("Need hh_id_obs or hh_id_obs_clean to count households.")

csb_panel_wave = (
    csb_agg_source
    .groupby(group_keys, as_index=False)
    .agg(agg_spec)
)

if "subsistence_agriculture" in csb_panel_wave.columns:
    csb_panel_wave = csb_panel_wave.rename(
        columns={"subsistence_agriculture": "avg_subsistence"}
    )

if "hh_id_obs_clean" in csb_panel_wave.columns:
    csb_panel_wave = csb_panel_wave.rename(
        columns={"hh_id_obs_clean": "n_HHID_wave"}
    )

if "hh_id_obs" in csb_panel_wave.columns:
    csb_panel_wave = csb_panel_wave.rename(
        columns={"hh_id_obs": "n_HHID_wave"}
    )

front_cols = [
    "Seed Bank",
    "CSB_clean",
    "Wave",
    "n_HHID_wave",
    "avg_subsistence",
    "AGSEC10_ANY",
    "AGSEC10_PROD",
    "AGSEC10_PRICES",
    "AGSEC10_PROC",
    "Mean_season_1",
    "Mean_season_2",
    "density_2010",
    "Trend",
    "share_hh_working_1",
    "share_hh_working_2",
]

front_cols = [c for c in front_cols if c in csb_panel_wave.columns]
other_cols = [c for c in csb_panel_wave.columns if c not in front_cols]

csb_panel_wave = csb_panel_wave[front_cols + other_cols]

print("CSB household-level expanded panel rows:", len(csb_panel))
print("CSB wave-level rows:", len(csb_panel_wave))
print("CSB wave-level columns:", len(csb_panel_wave.columns))

display(csb_panel_wave.head())

CSB HH list shape: (813, 4)


,Seed Bank,hh_id_obs,hh_id_obs_clean,CSB_clean
0,KIZIMBA CSB,7002841,7002841,KIZIMBA CSB
1,KIZIMBA CSB,7002842,7002842,KIZIMBA CSB
2,KIZIMBA CSB,7002843,7002843,KIZIMBA CSB
3,KIZIMBA CSB,7002844,7002844,KIZIMBA CSB
4,KIZIMBA CSB,7002845,7002845,KIZIMBA CSB



Merge diagnostic: HH panel by hh_id_obs
Left unique keys: 612
Right unique keys: 5654
_merge
both          611
left_only       1
right_only      0
Name: count, dtype: int64
CSB panel after HH merge: (3278, 26)


hh_panel_merge_status
both          3268
left_only       10
right_only       0
Name: count, dtype: int64

AGSEC10 merge status:


...

csb_rainfall_merge_status
both          3270
left_only        8
right_only       0
Name: count, dtype: int64


csb_density merge status:


csb_density_merge_status
both          3270
left_only        8
right_only       0
Name: count, dtype: int64

A3AQ38_1 outlier cells set to missing: 0
CSB household-level expanded panel rows: 3278
CSB wave-level rows: 115
CSB wave-level columns: 43


,Seed Bank,CSB_clean,Wave,n_HHID_wave,avg_subsistence,AGSEC10_ANY,AGSEC10_PROD,AGSEC10_PRICES,AGSEC10_PROC,Mean_season_1,...,A3Q26_1,A3Q26_2,A3AQ38_1,A3AQ38_2,A3Q39_1,A3Q39_2,A3Q41_1,A3Q41_2,A3Q43_1,A3Q43_2
0,ADIE CSB,ADIE CSB,1.0,79,0.088608,0.202532,1.000000,0.437500,0.125000,703.667606,...,0.025352,0.037806,4.306841,3.191908,45.414190,45.372969,0.619718,0.394366,128901.408451,38894.366197
1,ADIE CSB,ADIE CSB,2.0,71,0.253521,0.112676,1.000000,0.125000,0.000000,703.667606,...,0.025249,0.018242,3.137160,3.222453,34.779405,34.762703,0.406780,0.271186,46916.101695,44483.050847
2,ADIE CSB,ADIE CSB,3.0,74,0.162162,0.202703,0.933333,0.333333,0.066667,703.667606,...,0.012240,0.016553,3.057670,3.327702,35.430977,34.489481,0.281250,0.222222,44132.812500,16388.888889
3,ADIE CSB,ADIE CSB,4.0,44,0.295455,0.000000,NaN,NaN,NaN,703.667606,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ADIE CSB,ADIE CSB,5.0,46,0.23913,0.217391,0.800000,0.300000,0.200000,703.667606,...,0.015766,0.005714,2.746911,2.532857,60.597233,56.089899,0.324324,0.228571,20324.324324,17942.857143


In [19]:
OUT = Path(r"C:/Users/Carl/Desktop/CSB Baseline data")
out_file = OUT / "CSB_panel_v2.xlsx"

csb_panel_wave.to_excel(out_file, index=False)

print("Saved:", out_file)

Saved: C:\Users\Carl\Desktop\CSB Baseline data\CSB_panel_v2.xlsx
